In [8]:
import cv2
import numpy as np
from pathlib import Path
import random
from PIL import Image , ImageFilter


In [10]:
dataset_dir = Path('EALPR-master') / 'EALPR Vechicles dataset' / 'Vehicles'
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images = [
    path for path in dataset_dir.iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
]
print(len(images))
output_dir = dataset_dir/'output'

2087


In [11]:
sample_percentage = 10

In [12]:
NOISE_SIGMA_RANGE = (8, 25)
BLUR_RADIUS_RANGE = (0.5, 1.5)

RANDOM_SEED = 42

SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}


In [13]:
random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

In [14]:
def add_gaussian_noise(image):
    """Add random Gaussian noise to an image."""

    image_array = np.asarray(
        image.convert("RGB")
    ).astype(np.float32)

    sigma = random.uniform(*NOISE_SIGMA_RANGE)

    noise = rng.normal(
        loc=0,
        scale=sigma,
        size=image_array.shape
    )

    noisy_array = np.clip(
        image_array + noise, 0, 255
    ).astype(np.uint8)

    return Image.fromarray(noisy_array)


In [15]:
def add_gaussian_blur(image):
    """Apply Gaussian blur with a random radius."""

    radius = random.uniform(*BLUR_RADIUS_RANGE)

    return image.filter(
        ImageFilter.GaussianBlur(radius=radius)
    )

In [16]:
def augment_image(image):
    """Randomly choose noise, blur, or both."""

    operation = random.choice([
        "noise",
        "blur",
        "noise_blur"
    ])

    result = image.copy()

    if operation in ("noise", "noise_blur"):
        result = add_gaussian_noise(result)

    if operation in ("blur", "noise_blur"):
        result = add_gaussian_blur(result)

    return result, operation


In [17]:

def main():

    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"Dataset directory not found: {dataset_dir}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    # Only collect original images.
    # Never process images inside the output folder.
    image_paths = [
        path
        for path in dataset_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
        and output_dir not in path.parents
    ]

    if not image_paths:
        print("No images found.")
        return

    sample_count = max(
        1,
        round(len(image_paths) * sample_percentage / 100)
    )

    sample_count = min(sample_count, len(image_paths))

    selected_images = random.sample(
        image_paths, sample_count
    )

    saved_count = 0
    skipped_count = 0

    for image_path in selected_images:

        # Preserve subfolder structure and avoid filename collisions.
        relative_path = image_path.relative_to(dataset_dir)

        output_folder = output_dir / relative_path.parent
        output_folder.mkdir(parents=True, exist_ok=True)

        try:
            with Image.open(image_path) as original:
                original.load()
                image = original.convert("RGB")

            augmented_image, operation = augment_image(image)

            output_path = output_folder / (
                f"{image_path.stem}__aug_{operation}.png"
            )

            # Do not overwrite an existing augmented image.
            if output_path.exists():
                skipped_count += 1
                print(f"Skipped existing: {output_path}")
                continue

            augmented_image.save(output_path, format="PNG")

            saved_count += 1
            print(f"Saved: {output_path}")

        except Exception as error:
            print(f"Failed: {image_path} | {error}")

    print("\n========== SUMMARY ==========")
    print(f"Original images found: {len(image_paths)}")
    print(f"Randomly selected: {sample_count}")
    print(f"New images saved: {saved_count}")
    print(f"Existing images skipped: {skipped_count}")
    print(f"Output directory: {output_dir}")


if __name__ == "__main__":
    main()

Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0457__aug_noise_blur.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0103__aug_noise_blur.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\1127__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\1004__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0915__aug_noise_blur.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0572__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0420__aug_noise_blur.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0357__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\1729__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0131__aug_noise.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0123__aug_noise_blur.png
Saved: EALPR-master\EALPR Vechicles dataset\Vehicles\output\0384__aug_noise_blur.png
Saved: EAL